# Kinematic Guardrail Validation

This notebook validates the kinematic constraint verification system:
- Bone length invariance (L_bone)
- Biomechanical range of motion limits
- Temporal velocity limits
- Rigid body topology preservation
- Composite L_physics loss function

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('../../..')  # Add project root to path

import torch
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt

# Import our custom modules
from core.kinematic_guardrail import PoseEstimator, estimate_pose_from_image, COCO_KEYPOINTS, COCO_BONES
from core.identity_router import extract_arcface_embedding

print("Kinematic Guardrail Validation Notebook")
print("=======================================")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

In [ ]:
# Initialize pose estimator
print("Initializing Pose Estimator...")
try:
    pose_estimator = PoseEstimator(device="auto")
    print("✓ Pose estimator initialized successfully")
except Exception as e:
    print(f"✗ Failed to initialize pose estimator: {e}")
    print("Make sure controlnet-aux is installed: pip install controlnet-aux")

In [ ]:
# Test pose estimation on a sample image
print("Testing Pose Estimation...")

# Create a simple test image (you can replace this with actual test images)
test_image = np.zeros((512, 512, 3), dtype=np.uint8)
# Draw a simple stick figure for testing
cv2.circle(test_image, (256, 100), 5, (255, 255, 255), -1)  # nose
cv2.circle(test_image, (240, 90), 3, (255, 255, 255), -1)   # left eye
cv2.circle(test_image, (272, 90), 3, (255, 255, 255), -1)   # right eye
cv2.circle(test_image, (200, 120), 5, (255, 255, 255), -1)  # left shoulder
cv2.circle(test_image, (312, 120), 5, (255, 255, 255), -1)  # right shoulder
cv2.circle(test_image, (180, 200), 5, (255, 255, 255), -1)  # left elbow
cv2.circle(test_image, (332, 200), 5, (255, 255, 255), -1)  # right elbow
cv2.circle(test_image, (160, 280), 5, (255, 255, 255), -1)  # left wrist
cv2.circle(test_image, (352, 280), 5, (255, 255, 255), -1)  # right wrist
cv2.circle(test_image, (220, 250), 5, (255, 255, 255), -1)  # left hip
cv2.circle(test_image, (292, 250), 5, (255, 255, 255), -1)  # right hip
cv2.circle(test_image, (220, 350), 5, (255, 255, 255), -1)  # left knee
cv2.circle(test_image, (292, 350), 5, (255, 255, 255), -1)  # right knee
cv2.circle(test_image, (220, 450), 5, (255, 255, 255), -1)  # left ankle
cv2.circle(test_image, (292, 450), 5, (255, 255, 255), -1)  # right ankle

# Convert to PIL Image
test_pil = Image.fromarray(test_image)

try:
    # Estimate pose
    keypoints = pose_estimator.estimate_pose(test_pil)
    print(f"✓ Pose estimation successful. Keypoints shape: {keypoints.shape}")
    print(f"Keypoint confidence range: {keypoints[:, 2].min():.3f} - {keypoints[:, 2].max():.3f}")
    
    # Display keypoints
    print("\nDetected keypoints:")
    for i, (x, y, conf) in enumerate(keypoints):
        keypoint_name = COCO_KEYPOINTS.get(i, f"keypoint_{i}")
        print(f"  {i:2d}: {keypoint_name:12s} -> ({x:6.1f}, {y:6.1f}) conf: {conf:.3f}")
        
except Exception as e:
    print(f"✗ Pose estimation failed: {e}")

In [ ]:
# Test bone length calculation
print("Testing Bone Length Calculation...")

try:
    bone_lengths = pose_estimator.get_bone_lengths(keypoints)
    print("✓ Bone length calculation successful")
    
    print("\nBone lengths:")
    for bone_pair, length in bone_lengths.items():
        kp1_name = COCO_KEYPOINTS.get(bone_pair[0], f"kp_{bone_pair[0]}")
        kp2_name = COCO_KEYPOINTS.get(bone_pair[1], f"kp_{bone_pair[1]}")
        print(f"  {kp1_name} -> {kp2_name}: {length:.2f} pixels")
        
except Exception as e:
    print(f"✗ Bone length calculation failed: {e}")

In [ ]:
# Test joint angle calculation
print("Testing Joint Angle Calculation...")

try:
    joint_angles = pose_estimator.get_joint_angles(keypoints)
    print("✓ Joint angle calculation successful")
    
    print("\nJoint angles:")
    for joint_idx, angle in joint_angles.items():
        joint_name = COCO_KEYPOINTS.get(joint_idx, f"joint_{joint_idx}")
        print(f"  {joint_name}: {angle:.1f} degrees")
        
except Exception as e:
    print(f"✗ Joint angle calculation failed: {e}")

In [ ]:
# Test pose validation
print("Testing Pose Validation...")

try:
    validation_result = pose_estimator.validate_pose(keypoints)
    print("✓ Pose validation successful")
    
    print(f"\nValidation result: {'VALID' if validation_result['is_valid'] else 'INVALID'}")
    
    if validation_result['violations']:
        print("Violations found:")
        for violation in validation_result['violations']:
            print(f"  - {violation['type']}: {violation['joint']} angle {violation['angle']:.1f}° (limits: {violation['limits']})")
    else:
        print("No violations detected")
        
except Exception as e:
    print(f"✗ Pose validation failed: {e}")

In [ ]:
# Visualize pose estimation results
print("Creating Pose Visualization...")

try:
    # Create visualization
    vis_image = test_image.copy()
    
    # Draw keypoints
    for i, (x, y, conf) in enumerate(keypoints):
        if conf > 0.5:  # Only draw confident keypoints
            cv2.circle(vis_image, (int(x), int(y)), 4, (0, 255, 0), -1)
            cv2.putText(vis_image, str(i), (int(x)+5, int(y)-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # Draw bones
    for bone_pair in COCO_BONES:
        kp1 = keypoints[bone_pair[0]]
        kp2 = keypoints[bone_pair[1]]
        if kp1[2] > 0.5 and kp2[2] > 0.5:
            pt1 = (int(kp1[0]), int(kp1[1]))
            pt2 = (int(kp2[0]), int(kp2[1]))
            cv2.line(vis_image, pt1, pt2, (255, 0, 0), 2)
    
    # Display using matplotlib
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(vis_image, cv2.COLOR_BGR2RGB))
    plt.title("Pose Estimation Results - 17 Keypoint Skeleton")
    plt.axis('off')
    plt.show()
    
    print("✓ Pose visualization created successfully")
    
except Exception as e:
    print(f"✗ Pose visualization failed: {e}")

In [ ]:
# Test Velocity Loss (T2.4)
print("Testing Velocity Loss...")

from core.kinematic_guardrail import compute_velocity_loss, compute_l_physics, V_MAX_DEFAULT

print(f"v_max default: {V_MAX_DEFAULT} units/frame")

# Create sample pose keypoints: [T, K, 2] = [2 frames, 17 keypoints, x/y]
base = np.stack([np.linspace(0, 1, 17), np.linspace(1, 2, 17)], axis=-1).astype(np.float32)

# Test 1: Valid motion (velocity < v_max)
valid_pose = np.stack([base, base + [1.0, 0.0]], axis=0)[None]  # [B, T, K, 2]
velocities, loss = compute_velocity_loss(valid_pose, v_max=2.0)
print(f"Test 1 - Valid motion: max_velocity={np.max(velocities):.2f}, loss={loss:.4f}")

# Test 2: Invalid motion (velocity > v_max)
invalid_pose = np.stack([base, base + [3.0, 0.0]], axis=0)[None]  # velocity = 3.0 > v_max
velocities, loss = compute_velocity_loss(invalid_pose, v_max=2.0)
print(f"Test 2 - Invalid motion: max_velocity={np.max(velocities):.2f}, loss={loss:.4f}")

print("✓ Velocity loss tests passed")

In [ ]:
# Test L_physics Composite Loss
print("Testing L_physics Composite Loss...")

# Test with valid pose
result = compute_l_physics(valid_pose, v_max=2.0)
print(f"Valid pose: bone_loss={result['bone_loss']:.4f}, velocity_loss={result['velocity_loss']:.4f}, total={result['total_loss']:.4f}")

# Test with invalid pose
result = compute_l_physics(invalid_pose, v_max=2.0)
print(f"Invalid pose: bone_loss={result['bone_loss']:.4f}, velocity_loss={result['velocity_loss']:.4f}, total={result['total_loss']:.4f}")

# Test with custom weights
result = compute_l_physics(invalid_pose, v_max=2.0, bone_weight=0.5, velocity_weight=2.0)
print(f"Custom weights: total={result['total_loss']:.4f}")

print("✓ L_physics composite loss tests passed")

In [ ]:
# Test v_max Threshold Sensitivity
print("Testing v_max Threshold Sensitivity...")

# Create poses with different velocities
test_velocities = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
results = []

for vel in test_velocities:
    pose = np.stack([base, base + [vel, 0.0]], axis=0)[None]
    _, loss = compute_velocity_loss(pose, v_max=2.0)
    results.append((vel, loss))
    
# Display results
print(f"{'Velocity':<12} {'Loss':<10}")
print("-" * 22)
for vel, loss in results:
    status = "PASS" if loss == 0 else "FAIL"
    print(f"{vel:<12.1f} {loss:<10.4f} {status}")

print("✓ Threshold sensitivity test completed")